# DINOv3 Fine-tuning for Semantic Correspondence

This notebook performs fine-tuning of the DINOv3 model on the SPair-71k dataset for semantic correspondence tasks.

## What this notebook does:
- Loads the pre-trained DINOv3 ViT-B/16 model
- Sets up light fine-tuning (unfreezes only the last layer and norm)
- Trains on SPair-71k training set
- Validates on SPair-71k validation set with PCK metrics
- Saves the best model based on validation loss

## Requirements:
- Google Drive with datasets and model weights
- GPU recommended for training

## Expected runtime:
- ~30-60 minutes depending on NUM_EPOCHS setting

In [ ]:
%pip install torchmetrics                # ONLY FOR FIRST EXECUTION
%pip install git+https://github.com/facebookresearch/segment-anything.git

from google.colab import drive
import os

REPO_URL = "https://github.com/AML-Semantic-Correspondence/Semantic_Correspondence.git"

# 2. Clone/Pull the Code (access to logic)
print("\n📥 Setting up repository...")

# First ensure we're in a safe directory
%cd /content

# Clean up any existing problematic directories
if os.path.exists('/content/Semantic_Correspondence'):
    print("Removing existing Semantic_Correspondence directory...")
    !rm -rf /content/Semantic_Correspondence

if os.path.exists('/content/semantic-correspondence'):
    print("Removing existing semantic-correspondence directory...")
    !rm -rf /content/semantic-correspondence

# Clone the repository
print("Cloning repository fresh...")
try:
    !git clone {REPO_URL}
    
    # The repo will be cloned as 'Semantic_Correspondence', let's rename it for consistency
    if os.path.exists('/content/Semantic_Correspondence'):
        !mv /content/Semantic_Correspondence /content/semantic-correspondence
        print("✅ Repository cloned and renamed successfully")
    else:
        print("❌ Repository clone failed")
except Exception as e:
    print(f"❌ Error during clone: {e}")

# Mount drive and extract datasets
drive.mount("/content/drive", force_remount=True)
!tar -xzf "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/SPair-71k.tar.gz"
!unzip -o -q "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/PF-dataset-PASCAL.zip"
!unzip -o -q "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/PF-dataset.zip"

# Add the repository to path
%cd /content/semantic-correspondence
import sys
sys.path.append('/content/semantic-correspondence')

# Import training function
from src.training.train import train_step

# Run DINOv3 training
print("\n🚀 Starting DINOv3 fine-tuning...")
train_step('dinov3')
print("\n✅ DINOv3 training completed!")